### Part 3: The Logistics Commander (CoT & ToT) 

### 1. Initialization

In [ ]:
import pandas as pd
import time
import sys
import yaml
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.llm_client import LLMClient
from utils.prompts import render

# Load config to dynamically select the correct provider 
with open(project_root / "config" / "config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Initialize client using the mission-recommended 'reason' technique
client = LLMClient(provider=config['providers']['default'], technique="reason")

#Loading the data
with open("../data/incidents.txt", "r") as f:
    # Get rows but skip the header and separators
    incident_lines = [line.strip() for line in f if "|" in line and "ID" not in line and "---" not in line]

### 2. Step A (CoT - Scoring)

In [ ]:
scored_results = []
print("📝 Stage A: Executing Scoring...")

for row in incident_lines:
    prompt_text, _ = render("cot_scoring.v1", incident=row)
    res = client.chat([{"role": "user", "content": prompt_text}], 
                      task_type="classification", max_tokens=500, temperature=0.0)
    scored_results.append(f"ID: {row.split('|')[0].strip()} | {res['text']}")

all_scores_text = "\n".join(scored_results)

### 3. Step B (ToT - Strategy)


In [ ]:
print("🚢 Stage B: Strategic Route Selection...")

prompt_tot, _ = render("tot_strategy.v1", scored_incidents=all_scores_text)
res_strategy = client.chat([{"role": "user", "content": prompt_tot}], 
                          task_type="classification", max_tokens=1500, temperature=0.0)

### 4. Optimal Route Selection

In [ ]:
# The result below will now implicitly include the final "Optimal Route" line
print("\n" + "="*60)
print("COMMANDER'S LOG & MISSION DIRECTIVE")
print("="*60)
print(res_strategy['text'])